In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("/content/BostonHousing.csv")

# Check for missing values
print("Missing Values:\n", df.isnull().sum())

# Summary statistics
print("\nSummary Statistics:\n", df.describe())

# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
# the annotation would show the correlation values for each pair of variables in your DataFrame.
#annot=True, you're asking the heatmap to display the actual numeric value inside each colored cell.
plt.title("Feature Correlation Heatmap")

plt.show()

# Pairplot for key variables
sns.pairplot(df[['rm', 'lstat', 'ptratio', 'medv']])
plt.show()

# Distribution of target variable (MEDV)
plt.figure(figsize=(8, 5))
sns.histplot(df['medv'], bins=30, kde=True, color='blue')
plt.title("Distribution of House Prices (MEDV)")
plt.xlabel("MEDV")
plt.ylabel("Frequency")
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/BostonHousing.csv'

cmap="coolwarm":

This specifies the color map to use for the heatmap.

The coolwarm color map ranges from blue to red, where blue generally indicates negative correlations and red indicates positive correlations

**Interpretation**

MEDV (house prices) ranges from $5,000 to $50,000.

Average rooms per house (RM): ~6.28 (ranging from 3.56 to 8.78).

Crime rate (CRIM): Highly skewed, max value 88.97 while the median is 0.26.

Tax rate (TAX): Varies significantly, from 187 to 711.

The distribution is right-skewed, meaning some houses have very high prices.

There are peaks at 50, suggesting a potential price cap or outliers.

**INTERPRETATION OF PAIRPLOT**

RM vs MEDV: Clear positive linear relationship—houses with more rooms tend to have higher prices.

LSTAT vs MEDV: Strong negative correlation—higher % of lower-status residents leads to lower house prices.

CRIM vs MEDV: Houses in high-crime areas generally have lower prices, but some expensive houses exist even in high-crime regions.

PTRATIO vs MEDV: Weak negative correlation—higher student-to-teacher ratios slightly reduce house prices.

MODEL BUILDING

linear regresssion with original data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define features (X) and target (y)
X = df.drop(columns=["medv"])  # All columns except target
y = df["medv"]

# Split the dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model Evaluation
r2 = r2_score(y_test, y_pred)
#rmse = mean_squared_error(y_test, y_pred, squared=False)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


R² score (how well the model fits) and RMSE (error in prediction)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot of actual vs predicted values
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
plt.xlabel("Actual Prices (MEDV)")
plt.ylabel("Predicted Prices (MEDV)")
plt.title("Actual vs. Predicted House Prices")
plt.axline([0, 0], [1, 1], color="red", linestyle="--")  # 45-degree reference line
#This represents the first point on the line. It's the starting point with coordinates (x=0, y=0).
#This represents the second point on the line. It's the endpoint with coordinates (x=1, y=1).
#This code draws a red dashed line between the points (0, 0) and (1, 1) on the plot. The line will have a slope of 1 because it's going from (0,0) to (1,1)
plt.show()


If the points are closer to the red dashed line, predictions are accurate.

If there’s too much scatter, the model may need improvement (e.g., feature engineering, polynomial regression).

TO CHECK SKEWNESS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# Load the dataset
file_path = "/content/BostonHousing.csv"
df = pd.read_csv(file_path)

# Calculate skewness
skewness = df.skew()

# Plot histograms and boxplots
num_cols = len(df.columns)
plt.figure(figsize=(15, num_cols * 3))

for i, col in enumerate(df.columns):
    plt.subplot(num_cols, 2, 2 * i + 1)
    sns.histplot(df[col], kde=True)
    plt.title(f'Histogram of {col}\nSkewness: {skewness[col]:.2f}')

    plt.subplot(num_cols, 2, 2 * i + 2)
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')

plt.tight_layout()
plt.show()


In [ ]:
# Print skewness values for all numerical columns
skewness = df.skew()
print(skewness)


**Interpreting the Skewness Range:**

Skewness measures the asymmetry of a dataset's distribution. The range is interpreted as follows:

Symmetrical Distribution: Skewness ≈ 0
Moderate Skewness:
Negatively Skewed (Left-Skewed): -0.5 to -1

Positively Skewed (Right-Skewed): 0.5 to 1

**Handling Skweness**

If your dataset has skewed numerical features, you can apply transformations to make the distribution more normal, which often improves the performance of machine learning models.

Depending on whether the skew is positive (right-skewed) or negative (left-skewed), you can use different techniques:

1.Log Transformation (for right-skewed, positive values only):Use when data has large values and no zeros or negatives.

2.Square Root Transformation (for moderate right skew)
Works well when log transformation is too strong.

3.Box-Cox Transformation (for highly skewed, positive values)
Best for normalizing right-skewed distributions.

4.Yeo-Johnson Transformation (works for negative values too!)
Use when data has negative values or zeroes.


TRANSFORMATION OF SKEWNESS

In [ ]:

df.skew()

In [ ]:
import numpy as np

# Find highly skewed columns
skewed_cols = df.skew().abs()
highly_skewed = skewed_cols[skewed_cols > 1].index
negatively_skewed = skewed_cols[skewed_cols < 1].index
print("Highly Skewed Columns:\n", highly_skewed)
print("Negativly Skewed Columns:\n", negatively_skewed)


Log transformation

In [ ]:
df[highly_skewed] = df[highly_skewed].apply(lambda x: np.log1p(x))


In [ ]:
df[highly_skewed]

 Yeo-Johnson Transformation

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method='yeo-johnson')
df[negatively_skewed] = pt.fit_transform(df[negatively_skewed])


Check Skewness After Transformation

In [ ]:
print(df.skew())



**Which Method Should You Use?**
* Log/Square Root → If values are positive and right-skewed
* Box-Cox → If values are strictly positive and highly skewed
* Yeo-Johnson → If values include negative numbers or zero

LINEAR REGRESSION WITH SKEW TRANSFORMED DATA

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define features (X) and target (y)
X = df.drop(columns=["medv"])  # All columns except target
y = df["medv"]

# Split the dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model Evaluation
r2 = r2_score(y_test, y_pred)
#rmse = mean_squared_error(y_test, y_pred, squared=False)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


In [ ]:
# Define feature values manually
input_data = {
    "crim": float(input("Enter Crime Rate: ")),
    "zn": float(input("Enter Residential Land Zone: ")),
    "indus": float(input("Enter Non-Retail Business Acres: ")),
    "chas": int(input("Enter CHAS (0 or 1): ")),  # 1 if tract bounds river, else 0
    "nox": float(input("Enter NOX Concentration: ")),
    "rm": float(input("Enter Avg Rooms per Dwelling: ")),
    "age": float(input("Enter Age of Property: ")),
    "dis": float(input("Enter Distance to Employment Centers: ")),
    "rad": int(input("Enter Accessibility to Highways: ")),
    "tax": float(input("Enter Property Tax Rate: ")),
    "ptratio": float(input("Enter Pupil-Teacher Ratio: ")),
    "b": float(input("Enter Proportion of Black Residents: ")),
    "lstat": float(input("Enter Lower Status Population %: "))
}

# Convert to DataFrame
import pandas as pd
input_df = pd.DataFrame([input_data])

# Predict house price
predicted_price = model.predict(input_df)[0]

print(f" Predicted House Price (MEDV): {predicted_price:.2f}")


In [ ]:
df.head()

Appliying linear regression to skew transformed data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define new features (X) and target (y) using the transformed dataset
X_transformed = df.drop(columns=["medv"])  # Keep transformed features
y_transformed = df["medv"]

# Split the transformed dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y_transformed, test_size=0.2, random_state=42)

# Train Linear Regression Model with transformed features
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model Evaluation
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5  # Manually taking square root

print(f"📈 New R² Score: {r2:.4f}")
print(f"📉 New RMSE: {rmse:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot of actual vs predicted values
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
plt.xlabel("Actual Prices (MEDV)")
plt.ylabel("Predicted Prices (MEDV)")
plt.title("Actual vs. Predicted House Prices")
plt.axline([0, 0], [1, 1], color="red", linestyle="--")  # 45-degree reference line
#This represents the first point on the line. It's the starting point with coordinates (x=0, y=0).
#This represents the second point on the line. It's the endpoint with coordinates (x=1, y=1).
#This code draws a red dashed line between the points (0, 0) and (1, 1) on the plot. The line will have a slope of 1 because it's going from (0,0) to (1,1)
plt.show()


Some relationships might not be linear. Try polynomial regression to capture non-linear patterns.

linear regression with polynomial features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Apply Polynomial Features (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_transformed)  # Use transformed features

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y_transformed, test_size=0.2, random_state=42)

# Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions & Evaluation
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"🔹 Polynomial R² Score: {r2:.4f}")
print(f"🔹 Polynomial RMSE: {rmse:.4f}")


Applying Random forest model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions & Evaluation
y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f" Random Forest R² Score: {r2:.4f}")
print(f" Random Forest RMSE: {rmse:.4f}")
